# NOTE: Consider removing this step in the future and replace with code in model stream

In [2]:
import pandas as pd
import os
import numpy as np

# Get Home-Based Job percents from external process

In [3]:
wah_df = pd.read_csv(r"data\WAH-BaseYear-Chris-20251222\ControlTotal_WorkAtHome_2023.csv")
wah_df

,;Index,CO_FIPS,YEAR,Tel_RETL,Tel_FOOD,Tel_MANU,Tel_WSLE,Tel_OFFI,Tel_GVED,Tel_HLTH,...,HBJ_GVED,HBJ_HLTH,HBJ_OTHR,HBJ_AGRI,HBJ_MING,HBJ_CONS,HBJ,Telecom,Tot_WAH,CO_NAME
0,12023,1,2023,0.0444,0.0257,0.0424,0.0456,0.1306,0.0646,0.0474,...,0.0376,0.0276,0.0599,0.0234,0.0023,0.0226,0.0339,0.0583,0.0922,"Beaver County, Utah"
1,32023,3,2023,0.0531,0.0307,0.0507,0.0556,0.1563,0.0732,0.0567,...,0.0316,0.0245,0.0534,0.0196,0.0021,0.0200,0.0302,0.0699,0.1001,"Box Elder County, Utah"
2,52023,5,2023,0.0648,0.0375,0.0619,0.0753,0.1907,0.0914,0.0692,...,0.0251,0.0190,0.0417,0.0145,0.0016,0.0155,0.0277,0.1010,0.1287,"Cache County, Utah"
3,72023,7,2023,0.0436,0.0252,0.0416,0.0484,0.1284,0.0625,0.0466,...,0.0364,0.0271,0.0560,0.0187,0.0023,0.0222,0.0339,0.0583,0.0922,"Carbon County, Utah"
4,92023,9,2023,0.0446,0.0258,0.0425,0.0436,0.1311,0.0651,0.0476,...,0.0379,0.0277,0.0548,0.0248,0.0023,0.0227,0.0339,0.0583,0.0922,"Daggett County, Utah"
5,112023,11,2023,0.0847,0.0490,0.0808,0.0890,0.2491,0.1089,0.0904,...,0.0241,0.0200,0.0436,0.0141,0.0017,0.0164,0.0293,0.1323,0.1616,"Davis County, Utah"
6,132023,13,2023,0.0466,0.0270,0.0445,0.0460,0.1374,0.0679,0.0498,...,0.0395,0.0290,0.0621,0.0213,0.0024,0.0237,0.0339,0.0583,0.0922,"Duchesne County, Utah"
7,152023,15,2023,0.0446,0.0258,0.0426,0.0416,0.1316,0.0643,0.0477,...,0.0375,0.0278,0.0592,0.0248,0.0023,0.0227,0.0339,0.0583,0.0922,"Emery County, Utah"
8,172023,17,2023,0.0484,0.0280,0.0462,0.0538,0.1429,0.0695,0.0517,...,0.0405,0.0301,0.0609,0.0267,0.0025,0.0246,0.0339,0.0583,0.0922,"Garfield County, Utah"
9,192023,19,2023,0.0453,0.0262,0.0432,0.0512,0.1333,0.0642,0.0484,...,0.0374,0.0282,0.0525,0.0252,0.0024,0.0230,0.0339,0.0583,0.0922,"Grand County, Utah"


In [4]:
# FOR INCOMPLETE WAH DATA, SET WAH VALUES TO ZERO FOR ALL FORECAST YEARS

# 1. Identify unique counties
unique_fips = wah_df["CO_FIPS"].unique()

# 2. Determine existing year range and build new years
max_year = wah_df["YEAR"].max()
new_years = list(range(max_year + 1, 2066))   # 2065 inclusive

# If nothing to add, stop here
if len(new_years) > 0:

    # 3. Create a MultiIndex of all combinations
    new_index = pd.MultiIndex.from_product(
        [unique_fips, new_years],
        names=["CO_FIPS", "YEAR"]
    )

    # 4. Build an empty dataframe with those keys
    # Zero-fill all other numeric columns
    new_rows = pd.DataFrame(index=new_index).reset_index()

    # Identify numeric columns except CO_FIPS & YEAR
    num_cols = wah_df.select_dtypes(include="number").columns.difference(["CO_FIPS", "YEAR"])

    # Fill numeric with zeros
    for col in num_cols:
        new_rows[col] = 0

    # Add missing non-numeric columns too (if any)
    other_cols = wah_df.columns.difference(["CO_FIPS", "YEAR"] + list(num_cols))
    for col in other_cols:
        new_rows[col] = None

    # 5. Combine original + new rows
    wah_full_df = pd.concat([wah_df, new_rows], ignore_index=True)

    # 6. Sort neatly
    wah_full_df = wah_full_df.sort_values(["CO_FIPS", "YEAR"]).reset_index(drop=True)

else:
    wah_full_df = wah_df.copy()

wah_full_df['_Index'] = wah_full_df['CO_FIPS'] * 10000 + wah_full_df['YEAR']

wah_full_df


,;Index,CO_FIPS,YEAR,Tel_RETL,Tel_FOOD,Tel_MANU,Tel_WSLE,Tel_OFFI,Tel_GVED,Tel_HLTH,...,HBJ_HLTH,HBJ_OTHR,HBJ_AGRI,HBJ_MING,HBJ_CONS,HBJ,Telecom,Tot_WAH,CO_NAME,_Index
0,12023,1,2023,0.0444,0.0257,0.0424,0.0456,0.1306,0.0646,0.0474,...,0.0276,0.0599,0.0234,0.0023,0.0226,0.0339,0.0583,0.0922,"Beaver County, Utah",12023
1,0,1,2024,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,None,12024
2,0,1,2025,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,None,12025
3,0,1,2026,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,None,12026
4,0,1,2027,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,None,12027
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1285,0,1049,2061,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,None,10492061
1286,0,1049,2062,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,None,10492062
1287,0,1049,2063,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,None,10492063
1288,0,1049,2064,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,None,10492064


In [5]:
# normalize WAH data to long format

# get list of HBJ columns
hbj_cols = [c for c in wah_full_df.columns if c.startswith("HBJ_")]

# filter to only necessary columns
wah_hbj_df = wah_full_df[['CO_FIPS', 'YEAR'] + hbj_cols]

# melt the HBJ columns to long format
wah_melted_df = (
    wah_hbj_df
      .melt(
          id_vars=[c for c in wah_hbj_df.columns if not c.startswith("HBJ_")],
          value_vars=hbj_cols,
          var_name="HBJFULL",
          value_name="HBJPCT"
      )
      .assign(
          TDM_SECTOR=lambda d: d["HBJFULL"].str.split("_").str[1]
      )
      .drop(columns=["HBJFULL"])
).sort_values(by=['CO_FIPS', 'YEAR', 'TDM_SECTOR']).reset_index(drop=True)

# select and reorder final columns
wah_melted_df = wah_melted_df[['CO_FIPS', 'YEAR', 'TDM_SECTOR', 'HBJPCT']].rename(columns={'TDM_SECTOR':'VARIABLE'})

wah_melted_df

,CO_FIPS,YEAR,VARIABLE,HBJPCT
0,1,2023,AGRI,0.0234
1,1,2023,CONS,0.0226
2,1,2023,FOOD,0.0150
3,1,2023,GVED,0.0376
4,1,2023,HLTH,0.0276
...,...,...,...,...
14185,1049,2065,MING,0.0000
14186,1049,2065,OFFI,0.0000
14187,1049,2065,OTHR,0.0000
14188,1049,2065,RETL,0.0000


# Get employment data

In [6]:
tdm_employment_by_county_df = pd.read_csv('intermediate/split_emp.csv')
tdm_employment_by_county_df

,CO_FIPS_split,YEAR,VARIABLE,VALUE,CO_FIPS
0,1,2023,AGRI,738,1
1,1,2023,CONS,278,1
2,1,2023,FOOD,517,1
3,1,2023,GVED,810,1
4,1,2023,HLTH,160,1
...,...,...,...,...,...
15604,9157,2065,MING,164,57
15605,9157,2065,OFFI,14202,57
15606,9157,2065,OTHR,54646,57
15607,9157,2065,RETL,20356,57


# Join tables and calculate home-based jobs

In [7]:
tdm_hbj_df = pd.merge(
    tdm_employment_by_county_df,
    wah_melted_df,
    on=['CO_FIPS', 'YEAR', 'VARIABLE'],
    how='inner'
)
tdm_hbj_df

,CO_FIPS_split,YEAR,VARIABLE,VALUE,CO_FIPS,HBJPCT
0,1,2023,AGRI,738,1,0.0234
1,1,2023,CONS,278,1,0.0226
2,1,2023,FOOD,517,1,0.0150
3,1,2023,GVED,810,1,0.0376
4,1,2023,HLTH,160,1,0.0276
...,...,...,...,...,...,...
15604,9157,2065,MING,164,57,0.0000
15605,9157,2065,OFFI,14202,57,0.0000
15606,9157,2065,OTHR,54646,57,0.0000
15607,9157,2065,RETL,20356,57,0.0000


In [8]:
# do banker's rounding to calculate HBJ counts
tdm_hbj_df['HBJ'] = np.round(
    tdm_hbj_df['VALUE'] * tdm_hbj_df['HBJPCT']
).astype(int)

tdm_hbj_df['NONHBJ'] = (tdm_hbj_df['VALUE'] - tdm_hbj_df['HBJ']).astype(int)
tdm_hbj_df = tdm_hbj_df[['CO_FIPS_split','CO_FIPS','YEAR','VARIABLE','VALUE','HBJPCT','HBJ','NONHBJ']]
tdm_hbj_df

,CO_FIPS_split,CO_FIPS,YEAR,VARIABLE,VALUE,HBJPCT,HBJ,NONHBJ
0,1,1,2023,AGRI,738,0.0234,17,721
1,1,1,2023,CONS,278,0.0226,6,272
2,1,1,2023,FOOD,517,0.0150,8,509
3,1,1,2023,GVED,810,0.0376,30,780
4,1,1,2023,HLTH,160,0.0276,4,156
...,...,...,...,...,...,...,...,...
15604,9157,57,2065,MING,164,0.0000,0,164
15605,9157,57,2065,OFFI,14202,0.0000,0,14202
15606,9157,57,2065,OTHR,54646,0.0000,0,54646
15607,9157,57,2065,RETL,20356,0.0000,0,20356


In [8]:
tdm_hbj_df.to_csv('intermediate/split_emp_with_hbj_detail.csv', index=False)

## RUN CHECKS

In [9]:
# 1. Before HBJ: totals from tdm_employment_by_county_df
before_df = (
    tdm_employment_by_county_df
      .groupby(['CO_FIPS_split', 'YEAR'], as_index=False)
      .agg(VALUE_BEFORE_HBJ=('VALUE', 'sum'))
)

# 2. After HBJ: totals from tdm_hbj_df
after_df = (
    tdm_hbj_df
      .groupby(['CO_FIPS_split', 'YEAR'], as_index=False)
      .agg(
          VALUE_AFTER_HBJ=('HBJ', 'sum'),
          NONHBJ_AFTER_HBJ=('NONHBJ', 'sum')
      )
      .assign(
          TOTAL_AFTER_HBJ=lambda d: d['VALUE_AFTER_HBJ'] + d['NONHBJ_AFTER_HBJ']
      )
)

# 3. Combine and compute difference
tdm_hbj_check_df = (
    before_df
      .merge(after_df, on=['CO_FIPS_split', 'YEAR'], how='outer')
      .assign(
          DIFF=lambda d: d['VALUE_BEFORE_HBJ'] - d['TOTAL_AFTER_HBJ']
      )
)

# show where differences exist
tdm_hbj_check_df[tdm_hbj_check_df['DIFF'] != 0]


,CO_FIPS_split,YEAR,VALUE_BEFORE_HBJ,VALUE_AFTER_HBJ,NONHBJ_AFTER_HBJ,TOTAL_AFTER_HBJ,DIFF


### ONLY 2023 DATA FOR NOW

In [10]:
if tdm_hbj_check_df[(tdm_hbj_check_df['DIFF'] != 0)].size == 0:
    print("HBJ calculation check passed: No differences found for 2023 data.")
else:
    print("else")

HBJ calculation check passed: No differences found for 2023 data.


# Change to long format

In [11]:
long_nonhbj_df = tdm_hbj_df[['CO_FIPS_split','YEAR','VARIABLE','NONHBJ','CO_FIPS']].rename(columns={'NONHBJ':'VALUE'})
long_nonhbj_df

,CO_FIPS_split,YEAR,VARIABLE,VALUE,CO_FIPS
0,1,2023,AGRI,721,1
1,1,2023,CONS,272,1
2,1,2023,FOOD,509,1
3,1,2023,GVED,780,1
4,1,2023,HLTH,156,1
...,...,...,...,...,...
15604,9157,2065,MING,164,57
15605,9157,2065,OFFI,14202,57
15606,9157,2065,OTHR,54646,57
15607,9157,2065,RETL,20356,57


In [12]:
long_hbj_df = tdm_hbj_df.groupby(['CO_FIPS_split','YEAR','CO_FIPS'], as_index=False).agg(VALUE=('HBJ','sum'))
long_hbj_df['VARIABLE'] = 'HBJ'
long_hbj_df = long_hbj_df[['CO_FIPS_split','YEAR','VARIABLE','VALUE','CO_FIPS']]
long_hbj_df

,CO_FIPS_split,YEAR,VARIABLE,VALUE,CO_FIPS
0,1,2023,HBJ,159,1
1,1,2024,HBJ,0,1
2,1,2025,HBJ,0,1
3,1,2026,HBJ,0,1
4,1,2027,HBJ,0,1
...,...,...,...,...,...
1414,9157,2061,HBJ,0,57
1415,9157,2062,HBJ,0,57
1416,9157,2063,HBJ,0,57
1417,9157,2064,HBJ,0,57


In [13]:
long_df = pd.concat([long_hbj_df,long_nonhbj_df]).sort_values(['CO_FIPS_split','YEAR','VARIABLE'])
long_df

,CO_FIPS_split,YEAR,VARIABLE,VALUE,CO_FIPS
0,1,2023,AGRI,721,1
1,1,2023,CONS,272,1
2,1,2023,FOOD,509,1
3,1,2023,GVED,780,1
0,1,2023,HBJ,159,1
...,...,...,...,...,...
15604,9157,2065,MING,164,57
15605,9157,2065,OFFI,14202,57
15606,9157,2065,OTHR,54646,57
15607,9157,2065,RETL,20356,57


In [14]:
long_df.to_csv('intermediate/split_emp_with_hbj.csv', index=False)